# Golden Solution: Inventory Risk Assessment & Emergency Restocking

## Objective
Identify and rank the top 10 highest-risk inventory items requiring emergency restocking based on:
- Stockout probability calculation
- Safety stock violation detection
- Demand volatility analysis
- Deterministic multi-key ranking

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

print("✓ Libraries imported successfully")

## Section 1: Load & Validate Inventory Data

In [ ]:
# Initialize debug checks
debug_checks = {
    'data_loaded': False,
    'required_columns_exist': False,
    'no_null_numeric_values': False,
    'risk_scores_positive': False
}

required_columns = [
    'item_id', 'category', 'stock_level', 'reorder_point', 'daily_demand',
    'demand_std_dev', 'lead_time_days', 'holding_cost_per_unit_day', 
    'last_restock_date', 'stockout_count_last_month'
]

try:
    # Load data
    inventory_df = pd.read_csv('inventory_dataset.csv')
    debug_checks['data_loaded'] = True
    print(f"✓ Loaded {len(inventory_df)} inventory items")
    
    # Verify columns
    missing_cols = [c for c in required_columns if c not in inventory_df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns: {missing_cols}")
    debug_checks['required_columns_exist'] = True
    print("✓ All required columns present")
    
    # Check for nulls in numeric columns
    numeric_cols = ['stock_level', 'reorder_point', 'daily_demand', 'demand_std_dev', 'lead_time_days']
    if inventory_df[numeric_cols].isnull().any().any():
        raise ValueError("Null values found in numeric columns")
    debug_checks['no_null_numeric_values'] = True
    print("✓ No null values in numeric columns")
    
except Exception as e:
    print(f"✗ Data loading error: {e}")
    raise

## Section 2: Calculate Safety Stock & Risk Scores

In [ ]:
# Calculate safety stock baseline
inventory_df['safety_stock'] = (
    (inventory_df['daily_demand'] * inventory_df['lead_time_days']) +
    (2.33 * inventory_df['demand_std_dev'] * np.sqrt(inventory_df['lead_time_days']))
).round(2)

# Calculate days until stockout
inventory_df['days_until_stockout'] = np.maximum(1.0, inventory_df['stock_level'] / inventory_df['daily_demand']).round(1)

# Safety stock violation penalty
inventory_df['safety_violation'] = (inventory_df['stock_level'] < inventory_df['safety_stock']).astype(int)

# Demand volatility factor
inventory_df['demand_volatility'] = (inventory_df['demand_std_dev'] / inventory_df['daily_demand']).round(2)

# Calculate risk score: days_until_stockout × stockout_freq + safety_violation × 10 + volatility
inventory_df['risk_score'] = (
    (inventory_df['days_until_stockout'] * inventory_df['stockout_count_last_month']) +
    (inventory_df['safety_violation'] * 10.0) +
    inventory_df['demand_volatility']
).round(2)

# Verify all risk scores are non-negative
if (inventory_df['risk_score'] < 0).any():
    raise ValueError("Negative risk scores detected")
debug_checks['risk_scores_positive'] = True

print(f"✓ Risk scores calculated (range: {inventory_df['risk_score'].min()}-{inventory_df['risk_score'].max()})")
print(f"  Items at safety risk: {inventory_df['safety_violation'].sum()}")

## Section 3: Filter & Rank Emergency Items

In [ ]:
# Calculate 80th percentile risk threshold
percentile_80 = inventory_df['risk_score'].quantile(0.80)
print(f"✓ 80th percentile risk score: {percentile_80:.2f}")

# Apply multi-condition filter:
# 1. Risk >= 80th percentile
# 2. Proven stockout history (> 0)
# 3. Already below reorder point
critical_items = inventory_df[
    (inventory_df['risk_score'] >= percentile_80) &
    (inventory_df['stockout_count_last_month'] > 0) &
    (inventory_df['stock_level'] < inventory_df['reorder_point'])
].copy()

print(f"✓ Found {len(critical_items)} critical items meeting all criteria")

# Add urgency flags
critical_items['urgency_flag'] = critical_items.apply(
    lambda row: 'CRITICAL' if row['stock_level'] <= 0 else 'HIGH',
    axis=1
)

# Calculate recommended restock quantity
critical_items['recommended_restock_quantity'] = (
    (critical_items['reorder_point'] - critical_items['stock_level']) +
    (critical_items['daily_demand'] * 7)
).round(0)

# Sort with deterministic multi-key ordering:
# 1. risk_score DESC (highest risk first)
# 2. stock_level ASC (lowest stock first)
# 3. item_id ASC (lexicographic)
emergency_restock_df = critical_items.sort_values(
    by=['risk_score', 'stock_level', 'item_id'],
    ascending=[False, True, True]
).head(10).copy()

emergency_item_ids = emergency_restock_df['item_id'].astype(str).tolist()

print(f"✓ Ranked emergency items: {len(emergency_item_ids)} items")
print(f"  Top 3: {emergency_item_ids[:3]}")

## Section 4: Export Results

In [ ]:
# Create output directory
os.makedirs('/content', exist_ok=True)

# Calculate summary metrics
total_emergency_stock_deficit = emergency_restock_df['recommended_restock_quantity'].sum()
highest_risk_score = emergency_restock_df['risk_score'].max() if len(emergency_restock_df) > 0 else 0.0
total_emergency_stock_deficit = round(float(total_emergency_stock_deficit), 2)
highest_risk_score = round(float(highest_risk_score), 2)

# Export audit CSV
export_columns = ['item_id', 'category', 'stock_level', 'reorder_point', 
                  'risk_score', 'urgency_flag', 'days_until_stockout', 
                  'recommended_restock_quantity']
audit_df = emergency_restock_df[export_columns].copy()
csv_path = '/content/emergency_restock_audit.csv'
audit_df.to_csv(csv_path, index=False)
print(f"✓ Exported audit CSV: {csv_path}")

# Create JSON output
strategic_plan = {
    'emergency_item_ids': emergency_item_ids,
    'total_emergency_stock_deficit': total_emergency_stock_deficit,
    'highest_risk_score': highest_risk_score,
    'debug_checks': debug_checks
}

json_path = '/content/emergency_action_plan.json'
with open(json_path, 'w') as f:
    json.dump(strategic_plan, f, indent=2)
print(f"✓ Exported action plan JSON: {json_path}")

# Verification
print("\n" + "="*60)
print("EXECUTION VERIFICATION")
print("="*60)
print(json.dumps(strategic_plan, indent=2))